# MMS Poisson — Method of Manufactured Solutions

**Poisson equation on the unit square**: −Δu = f, with weak Dirichlet BC.

**Manufactured solution**: u = sin(πx)sin(πy) + xy (non-zero boundary data).

**Scheme**: P1 Lagrange, residual form + Newton, structured triangular meshes N×N,
N = 8, 16, 32, 64, 128.

**Reference rates**: L2 → 2, H1 → 1 (optimal for P1).

Two BC methods are compared:
- **penalty**: γ(u−g, v)_∂Ω, γ = C/h — non-symmetric, consistency error O(h/C)
- **nitsche**: symmetric Nitsche — (∇u·n, v) + (∇v·n, u−g) + γ(u−g, v) — optimal for all C above threshold

Errors measured against exact solution via UFL SpatialCoordinate expressions;
f and g enter the forms directly as UFL (no nodal interpolation);
quadrature degree = 2k+3 = 5 (k = 1 for P1).

In [ ]:
# --- canonical: bootstrap v1 ---
import sys, os, subprocess

_on_colab = "google.colab" in sys.modules

if not _on_colab:
    os.environ.setdefault("OMP_NUM_THREADS", "1")

if _on_colab:
    try:
        import gmsh
    except ImportError:
        subprocess.run(
            'wget -q "https://fem-on-colab.github.io/releases/gmsh-install.sh"'
            ' -O /tmp/gmsh-install.sh && bash /tmp/gmsh-install.sh',
            shell=True, check=True,
        )
    try:
        import dolfinx
    except ImportError:
        subprocess.run(
            'wget -q "https://fem-on-colab.github.io/releases/fenicsx-install-release-real.sh"'
            ' -O /tmp/fenicsx-install.sh && bash /tmp/fenicsx-install.sh',
            shell=True, check=True,
        )

import dolfinx
print("dolfinx version:", dolfinx.__version__)

if _on_colab:
    from google.colab import files
# --- end canonical: bootstrap ---

In [ ]:
import os, math
from mpi4py import MPI
import ufl
from petsc4py import PETSc
from dolfinx.mesh import create_unit_square, CellType
from dolfinx.fem import functionspace, Function, Constant, form, assemble_scalar
from dolfinx.fem.petsc import NonlinearProblem
import basix.ufl as bufl


In [ ]:
# DD2365_FAST: run N=8,16 only for CI
N_list = [8, 16] if os.environ.get('DD2365_FAST') == '1' else [8, 16, 32, 64, 128]
C_penalty = 1e3
C_nitsche_list = [10, 100, 1000]


## Manufactured solution

u_ex = sin(πx)sin(πy) + xy  →  f = −Δu_ex = 2π²sin(πx)sin(πy)

Boundary data g = u_ex|∂Ω.

Both f and g enter the weak form as UFL SpatialCoordinate expressions — no nodal interpolation.

## Convergence study

**Penalty form** (non-symmetric):

    F(u; v) = ∫∇u·∇v dx + γ∫(u − g)v ds − ∫f·v dx = 0,   γ = C/h

**Symmetric Nitsche form**:

    F(u; v) = ∫∇u·∇v dx
            − ⟨∇u·n, v⟩_∂Ω          (consistency)
            − ⟨∇v·n, u−g⟩_∂Ω        (symmetry)
            + γ⟨u−g, v⟩_∂Ω          (penalty)
            − ∫f·v dx = 0,   γ = C/h

Coercivity of Nitsche requires C > C_inv ≈ 5 for P1 on regular triangles.
Solved with NonlinearProblem + Newton (1 step — linear).

Error columns: e_L2 = ‖u_h−u_ex‖_L2(Ω), e_H1 = ‖∇(u_h−u_ex)‖_L2(Ω),
e_bdy = ‖u_h−g‖_L2(∂Ω).  Quadrature degree 5 (= 2·1+3).

In [ ]:
def run_poisson(N_list, C, method='penalty'):
    """method: 'penalty' or 'nitsche'"""
    pi    = math.pi
    qdeg  = 5  # 2*k+3, k=1 for P1
    dx_ex = ufl.dx(metadata={'quadrature_degree': qdeg})
    ds_ex = ufl.ds(metadata={'quadrature_degree': qdeg})
    results = []
    for N in N_list:
        msh = create_unit_square(MPI.COMM_WORLD, N, N, cell_type=CellType.triangle)
        V   = functionspace(msh, bufl.element('Lagrange', msh.basix_cell(), 1))
        u   = Function(V)
        v   = ufl.TestFunction(V)

        h_c   = ufl.CellDiameter(msh)
        gamma = Constant(msh, PETSc.ScalarType(C)) / h_c

        x     = ufl.SpatialCoordinate(msh)
        u_ex  = ufl.sin(pi*x[0])*ufl.sin(pi*x[1]) + x[0]*x[1]
        f_ufl = 2.0*pi**2 * ufl.sin(pi*x[0])*ufl.sin(pi*x[1])

        if method == 'nitsche':
            n = ufl.FacetNormal(msh)
            F = (
                ufl.inner(ufl.grad(u), ufl.grad(v)) * ufl.dx
                - ufl.inner(ufl.grad(u), n) * v * ufl.ds
                - ufl.inner(ufl.grad(v), n) * (u - u_ex) * ufl.ds
                + gamma * (u - u_ex) * v * ufl.ds
                - f_ufl * v * ufl.dx
            )
        else:
            F = (
                ufl.inner(ufl.grad(u), ufl.grad(v)) * ufl.dx
                + gamma * (u - u_ex) * v * ufl.ds
                - f_ufl * v * ufl.dx
            )

        problem = NonlinearProblem(
            F, u,
            petsc_options_prefix='mms_poisson_',
            petsc_options={
                'snes_type': 'newtonls',
                'snes_rtol': 1e-12,
                'snes_atol': 1e-14,
                'ksp_type': 'preonly',
                'pc_type': 'lu',
                'pc_factor_mat_solver_type': 'mumps',
            },
        )
        problem.solve()
        n_iters = problem.solver.getIterationNumber()

        e     = u - u_ex
        e_L2  = math.sqrt(abs(float(msh.comm.allreduce(
            assemble_scalar(form(e**2 * dx_ex)), op=MPI.SUM))))
        e_H1  = math.sqrt(abs(float(msh.comm.allreduce(
            assemble_scalar(form(ufl.inner(ufl.grad(e), ufl.grad(e)) * dx_ex)), op=MPI.SUM))))
        e_bdy = math.sqrt(abs(float(msh.comm.allreduce(
            assemble_scalar(form(e**2 * ds_ex)), op=MPI.SUM))))

        h_val = 1.0 / N
        results.append({'N': N, 'h': h_val,
                         'e_L2': e_L2, 'e_H1': e_H1, 'e_bdy': e_bdy,
                         'iters': n_iters})
        print(f'N={N:4d}  h={h_val:.4f}  e_L2={e_L2:.4e}  e_H1={e_H1:.4e}  '
              f'e_bdy={e_bdy:.4e}  iters={n_iters}')
    return results


In [ ]:
print(f'C = {C_penalty:.0e}')
results_default = run_poisson(N_list, C_penalty)


### Results table

In [ ]:
def print_table(results, label=''):
    if label:
        print(f'--- {label} ---')
    hdr = ('N'.rjust(6) + '  ' + 'h'.rjust(8) + '  ' +
           'e_L2'.rjust(12) + '  ' + 'r_L2'.rjust(6) + '  ' +
           'e_H1'.rjust(12) + '  ' + 'r_H1'.rjust(6) + '  ' +
           'e_bdy'.rjust(12) + '  ' + 'r_bdy'.rjust(6))
    print(hdr)
    for i, r in enumerate(results):
        if i == 0:
            rl2 = rh1 = rb = '    —'
        else:
            prev = results[i-1]
            rl2 = f'{math.log2(prev["e_L2" ]/r["e_L2" ]):6.2f}' if r['e_L2' ] > 0 else '  NaN'
            rh1 = f'{math.log2(prev["e_H1" ]/r["e_H1" ]):6.2f}' if r['e_H1' ] > 0 else '  NaN'
            rb  = f'{math.log2(prev["e_bdy"]/r["e_bdy"]):6.2f}' if r['e_bdy'] > 0 else '  NaN'
        print(f"{r['N']:>6}  {r['h']:>8.4f}  {r['e_L2']:>12.4e}  {rl2}  "
              f"{r['e_H1']:>12.4e}  {rh1}  {r['e_bdy']:>12.4e}  {rb}")

print_table(results_default, f'C = {C_penalty:.0e}')


### Penalty sensitivity

In [ ]:
print('\n=== C = 1e1 ===')
results_c1 = run_poisson(N_list, 1e1)
print('\n=== C = 1e5 ===')
results_c5 = run_poisson(N_list, 1e5)


In [ ]:
print_table(results_c1, 'C = 1e1')
print()
print_table(results_c5, 'C = 1e5')


## Nitsche method

Symmetric Nitsche BC: three terms — consistency, symmetry, penalty.
Coercivity threshold: C_inv ≈ 5 for P1 on regular triangles.
Expected: optimal rates (L2→2, H1→1) for all C > C_inv, including C=10.
e_bdy should decrease at rate ≥ 1 (H1 boundary norm); much smaller than penalty at same C.

In [ ]:
nitsche_results = {}
for C_n in C_nitsche_list:
    print(f'\n=== Nitsche C = {C_n} ===')
    nitsche_results[C_n] = run_poisson(N_list, C_n, method='nitsche')


In [ ]:
for C_n in C_nitsche_list:
    print()
    print_table(nitsche_results[C_n], f'Nitsche C = {C_n}')
